# Notebook 2: Teach a DNA model to spot promoters

**Framework:** Hugging Face Transformers + PEFT · **Model:** `InstaDeepAI/nucleotide-transformer-500m-human-ref` · **Time:** ~60 minutes · **GPU:** free Colab T4

In Notebook 1 the model learned a *style*. Here it learns a *task*: reading a 300-letter DNA sequence and deciding whether it is a **promoter** (a stretch of DNA that switches a gene on) or not. Instead of generating text, the model outputs one of two labels, so we can measure exactly how often it's right.

**What you'll do**
1. Load a DNA language model and add a classification head to it.
2. Prepare the data: tokenise DNA and make training, validation and test sets.
3. Add LoRA adapters.
4. Measure the model **before** training, and try a cheaper alternative: training only the head.
5. Train with LoRA.
6. Compare **before vs after** with accuracy, F1 and MCC, and look at the mistakes.
7. Save your adapter.

**Learning outcomes covered:** 2 (preparing data), 3 (LoRA), 4 (evaluation).

> **DRAFT NOTE (facilitator timing, remove before publishing):**
> 0 Install ~2 min · 1 Load ~5 min (download runs while you introduce the task) · 2 Data ~12 min · 3 LoRA ~5 min · 4 Baseline ~10 min (≈1 min running) · 5 Train ~8 min (≈3 min running) · 6 Evaluate ~12 min · 7 Save ~2 min · buffer ~4 min.

### Same method, different framework

LoRA is a general technique, not an Unsloth feature. Unsloth is mainly built for chat models, so for this DNA model we use the Hugging Face libraries directly: `transformers` for the model and the `Trainer`, and `peft` for LoRA. Unsloth is built on these same libraries.

The sections match Notebook 1. What changes is the task:

| | Notebook 1: Pirate | Notebook 2: Promoters |
|---|---|---|
| **Load** | a chat model that generates text | a model plus a new **classification head** |
| **Data** | conversations in a chat template | DNA sequences with a 0/1 label |
| **LoRA** | `FastModel.get_peft_model`: name the *parts* to train, Unsloth finds the layers | `peft.get_peft_model`: name the layers yourself |
| **Train** | `SFTTrainer` (TRL) | `Trainer` (Transformers) |
| **Evaluate** | read the answers, count pirate words | compute metrics on a labelled test set |

### Before you start: turn on the GPU
In Colab, go to **Runtime → Change runtime type**, choose **T4 GPU**, and click **Save**. The cell below checks that it worked.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Install

Colab already has most of what we need. This installs **PEFT** (the LoRA library) and makes sure `transformers`, `datasets` and `accelerate` are recent enough.

> **DRAFT NOTE:** After the T4 test, run `!pip freeze | grep -iE "transformers|peft|datasets|accelerate|torch|scikit-learn"` and pin the versions that worked. The model ships as `pytorch_model.bin` (no safetensors), which recent `transformers` only loads with `torch>=2.6`; Colab's torch should be newer than that, but check.

In [ ]:
%%capture
!pip install -U peft transformers datasets accelerate

## 1. Load the base model

The **Nucleotide Transformer** is a language model trained on DNA instead of text. This version has 500 million parameters and was pre-trained on the human reference genome. Like a text model learning grammar by predicting missing words, it learned patterns in DNA by predicting missing pieces of sequence. It was never told what a promoter is: pre-training teaches it the "language" of DNA, and fine-tuning teaches it the question we're asking.

We load it with `AutoModelForSequenceClassification`, which adds a **classification head**: a small new layer on top of the model that turns its output into a score for each label (0 = not a promoter, 1 = promoter).

- The head is new, so it starts with **random weights**. The load report lists the `classifier` weights as MISSING (newly created) and the `lm_head` weights as UNEXPECTED (the part used in pre-training to predict missing DNA, which we don't need). Both are expected.
- We'll load the model twice in this notebook (once more in section 4b), so we put the loading in a function. Setting the seed first means the random head is identical each time.

**Licence:** this model is CC BY-NC-SA 4.0 (non-commercial use only). Always check the licence of a model before building on it.

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed

SEED = 3407
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
LABELS = ["not promoter", "promoter"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_base_model():
    set_seed(SEED)   # same random head every time
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=dict(enumerate(LABELS)),
        label2id={label: i for i, label in enumerate(LABELS)},
    ).to("cuda")

model = load_base_model()

## 2. Prepare the data

We use the **promoter** task from InstaDeep's genomics benchmark. Each example is a 300-letter DNA sequence (A, C, G, T) from the human genome with a label: 1 if it contains a promoter, 0 if it doesn't.

The data comes as two files: 30,000 sequences for training and 1,584 for testing. The test sequences come from **different chromosomes** from the training sequences, so the model can't score well by recognising nearby or overlapping DNA it trained on.

> **DRAFT NOTE:** The `_revised` dataset has no `promoter_all` config on the Hub (`load_dataset(name)` loads all 18 tasks at once), so we load the parquet files directly. Also check and record the dataset licence: the card doesn't state one, and the promoters come from the Eukaryotic Promoter Database.

In [ ]:
from datasets import load_dataset

DATA_PATH = "hf://datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised/promoter_all"
data = load_dataset("parquet", data_files={"train": f"{DATA_PATH}/train.parquet",
                                           "test": f"{DATA_PATH}/test.parquet"})
print(data)
data["train"][0]

### 2a. Look at the data

Before training anything, check the basics: how long the sequences are, and how many of each label there are. If one label were much more common, a model could score well just by always guessing it.

In [ ]:
import pandas as pd

for name, split in data.items():
    labels = pd.Series(split["label"]).map(dict(enumerate(LABELS)))
    lengths = pd.Series(split["sequence"]).str.len()
    print(f"{name}: {len(split)} sequences, lengths {lengths.min()}–{lengths.max()}")
    print(labels.value_counts().to_string(), "\n")

The labels are balanced (about 50/50), so always guessing one label would score about 50% accuracy. That's the score to beat.

### 2b. Training, validation and test sets

Training on all 30,000 sequences would take too long here, so we take a random sample. We use the same three sets as in Notebook 1:

| Set | Size | Used for |
|---|---|---|
| **Training** | 5,000 | The model learns from these. |
| **Validation** | 1,000 | Checked *during* training, to spot overfitting. Taken from the training file. |
| **Test** | 1,584 | Used only for the before/after evaluation. The model never sees them during training. |

The test set must stay untouched. If you check test scores while you adjust settings, you end up choosing the settings that suit the test set, and the final score is no longer a fair estimate.

In [ ]:
N_TRAIN, N_VAL = 5_000, 1_000   # try 10_000 training sequences if you have time

sample = data["train"].shuffle(seed=SEED).select(range(N_TRAIN + N_VAL))
split = sample.train_test_split(test_size=N_VAL, seed=SEED)
train_ds, val_ds = split["train"], split["test"]
test_ds = data["test"]

print(f"Training: {len(train_ds)}   Validation: {len(val_ds)}   Test: {len(test_ds)}")

### 2c. Tokenising DNA

A text model splits sentences into **tokens** (words or pieces of words). This model splits DNA into **6-mers**: chunks of 6 letters, such as `ACGTGA`. There are 4⁶ = 4,096 possible 6-mers, and each one has its own number in the model's vocabulary.

A 300-letter sequence becomes 50 tokens, plus a special `<cls>` token at the start that the classification head reads from.

In [ ]:
seq = train_ds[0]["sequence"]
tokens = tokenizer.tokenize(seq)
print(seq[:60], "...")
print(tokens[:10], "...")
print(f"{len(seq)} letters -> {len(tokens)} tokens")

ids = tokenizer(seq)["input_ids"]
print(f"With special tokens: {len(ids)} ids, starting {tokenizer.convert_ids_to_tokens(ids[:3])}")

**Question:** What do you think the tokenizer does with a sequence whose length isn't a multiple of 6, or one that contains an `N` (an unknown base)? Try `tokenizer.tokenize("ACGTACGTN")`.

Now tokenise every sequence. The `Trainer` only needs the token ids and the label, so we drop the other columns. We keep the untokenised `test_ds` for evaluation in sections 4 and 6.

In [ ]:
def tokenize(batch):
    return tokenizer(batch["sequence"])

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["sequence", "name", "task"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["sequence", "name", "task"])
train_tok

## 3. Add LoRA adapters

Same idea as Notebook 1: freeze the original model and train small adapter matrices alongside some of its layers. The settings are almost the same. The differences:

- **We name the layers ourselves.** In Notebook 1 we told Unsloth which *parts* to train and it found the layers. Plain PEFT needs the layer names in `target_modules`. In this model the attention layers are called `query`, `key` and `value` (run `print(model)` to see them). We adapt `query` and `value`, a common choice.
- `task_type=TaskType.SEQ_CLS` tells PEFT this is a classification model. PEFT then also trains the new **classification head** in full, because it started random and has to be learned from scratch.
- `lora_alpha` scales how strongly the adapters affect the model. Notebook 1 set it equal to `r`; twice `r` is also common. It's a setting worth tuning.
- `lora_dropout` is 0.1 here rather than 0. Unsloth runs faster with 0; plain PEFT gets no such speed-up, so we use a little dropout to reduce overfitting.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                               # adapter size (rank); try 8 or 32
    lora_alpha=32,                      # 2 x r; try 16 (= r, as in Notebook 1)
    lora_dropout=0.1,                   # try 0 or 0.05
    target_modules=["query", "value"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

**Question:** Compare the percentage of trained parameters with Notebook 1. Part of this count is the classification head. Which layers do you think make up the rest?

In [ ]:
# Which kinds of layer got adapters? (Each LoRA adapter adds a "lora_A" weight next to its layer.)
sorted({name.split(".")[-4] for name, _ in model.named_parameters() if "lora_A" in name})

You should see only `query` and `value`. The classification head isn't listed because it has no adapter: it's trained in full.

## 4. Baseline: how does the model do before training?

For a classification task we can measure performance exactly. We run the model on the test set, turn its outputs into a **probability** that each sequence is a promoter, and predict "promoter" when that probability is above 0.5.

We report three **metrics** (scores):

- **Accuracy:** the fraction of predictions that are correct.
- **F1:** balances missed promoters against false alarms. Ranges from 0 to 1.
- **MCC** (Matthews correlation coefficient): ranges from −1 to 1, where 0 means no better than guessing and 1 means perfect. It stays honest when one label is much more common than the other, so it's the standard metric in genomics benchmarks.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

TEST_SEQUENCES = list(test_ds["sequence"])
TEST_LABELS = np.array(test_ds["label"])

def predict(m, sequences, batch_size=64):
    # Returns the probability of "promoter" for each sequence
    m.eval()
    probs = []
    for i in range(0, len(sequences), batch_size):
        batch = tokenizer(sequences[i:i + batch_size], return_tensors="pt", padding=True).to(m.device)
        with torch.no_grad():
            logits = m(**batch).logits
        probs += torch.softmax(logits.float(), dim=-1)[:, 1].tolist()
    return np.array(probs)

def metrics(labels, preds):
    return {"accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds),
            "mcc": matthews_corrcoef(labels, preds)}

def evaluate(m):
    probs = predict(m, TEST_SEQUENCES)
    preds = (probs > 0.5).astype(int)
    scores = metrics(TEST_LABELS, preds)
    print("   ".join(f"{k}: {v:.3f}" for k, v in scores.items()))
    return probs, preds, scores

### 4a. The untrained model

The head is random, so expect accuracy near 0.5 and MCC near 0, i.e. guessing. The model may even predict the same label for almost everything.

In [ ]:
before_probs, before_preds, before_scores = evaluate(model)
print("Predicted labels:", np.bincount(before_preds, minlength=2))

### 4b. A cheaper alternative: train only the head

In Notebook 1, the cheap alternative to fine-tuning was a system prompt. A classification model can't take a prompt, but there's a different cheap option: **freeze the whole pre-trained model and train only the head** (sometimes called a *linear probe*). This tests how much the pre-trained model already "knows" about promoters without changing any of its weights.

We load a fresh copy of the model and freeze everything except the head:

In [ ]:
head_only_model = load_base_model()
for name, param in head_only_model.named_parameters():
    param.requires_grad = name.startswith("classifier")

n_trainable = sum(p.numel() for p in head_only_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in head_only_model.parameters())
print(f"trainable params: {n_trainable:,} || all params: {n_total:,} || trainable%: {100 * n_trainable / n_total:.4f}")

We train with Hugging Face's `Trainer`. `TrainingArguments` holds the settings. We'll reuse **exactly the same settings** for LoRA in section 5, so the comparison is fair.

- `compute_metrics` reports accuracy, F1 and MCC on the validation set during training.
- `fp16=True` uses 16-bit maths where possible, which makes training faster on the T4.
- One **epoch** means one pass through all 5,000 training sequences.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metrics(labels, np.argmax(logits, axis=-1))

training_args = TrainingArguments(
    output_dir="outputs",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=5e-4,              # TODO: tune after T4 test; try 1e-4 or 1e-3
    warmup_steps=15,                 # ~5% of the 313 steps
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    seed=SEED,
    report_to="none",
)

def make_trainer(m):
    return Trainer(
        model=m,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=DataCollatorWithPadding(tokenizer),   # batches the token ids together
        compute_metrics=compute_metrics,
    )

Train the head only. This takes about a minute, because no gradients flow through the frozen model.

> **DRAFT NOTE:** Time this on a T4.

In [ ]:
head_only_trainer = make_trainer(head_only_model)
head_only_trainer.train()
head_only_probs, head_only_preds, head_only_scores = evaluate(head_only_model)

**Question:** Is this better than guessing? If training only the head already worked well, would you bother with LoRA?

We're done with this copy of the model, so we delete it to free GPU memory.

In [ ]:
import gc

del head_only_model, head_only_trainer
gc.collect()
torch.cuda.empty_cache()

## 5. Train with LoRA

Now train the LoRA model with the same settings. This time the adapters inside the model learn too, so the model can adjust how it reads DNA for this task, not just how the head interprets it.

This takes about 3 minutes on a T4. Watch the **training loss** go down, and the validation **MCC** go up every 100 steps.

> **DRAFT NOTE:** Time this on a T4 and adjust `N_TRAIN` (target ~3–5 min). ahoy_training reached accuracy 0.887 and MCC 0.774 on this task; record what we get here.

In [ ]:
trainer = make_trainer(model)
stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime']:.0f} seconds")

### Plot the loss

As in Notebook 1: if the training loss keeps falling but the validation loss starts rising, the model is **overfitting**.

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train = [(l["step"], l["loss"]) for l in logs if "loss" in l]
val = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.plot(*zip(*train), label="training loss")
plt.plot(*zip(*val), "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

> **Training failed or ran out of time?** Run the cell below to load an adapter we trained before the workshop, then carry on with section 6.

In [ ]:
# Only run this if your training didn't finish.
# TODO: upload the fallback adapter and set its name here
# from peft import PeftModel
# model = PeftModel.from_pretrained(load_base_model(), "UoA-eResearch/promoter-nt500m-lora")

## 6. Evaluate: before vs after

Run the trained model on the same test set as in section 4.

In [ ]:
after_probs, after_preds, after_scores = evaluate(model)

### 6a. Compare the scores

All three models were scored on the same 1,584 test sequences, from chromosomes none of them trained on.

In [ ]:
pd.DataFrame([before_scores, head_only_scores, after_scores],
             index=["before (random head)", "head only", "LoRA"]).round(3)

**Think about it:** How much did LoRA add over training only the head? This model was pre-trained on DNA but never told what a promoter is. What does the head-only score tell you about what it learned during pre-training?

### 6b. What kind of mistakes?

A **confusion matrix** counts each combination of true and predicted label. The diagonal holds the correct predictions. The other two cells are the two kinds of mistake: **missed promoters** (false negatives) and **false alarms** (false positives).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, preds, title in zip(axes, [head_only_preds, after_preds], ["head only", "LoRA"]):
    ConfusionMatrixDisplay.from_predictions(TEST_LABELS, preds, display_labels=LABELS,
                                            ax=ax, colorbar=False)
    ax.set_title(title)
plt.tight_layout(); plt.show()

**Think about it:** In your own research, would a missed promoter or a false alarm be more costly? You can trade one against the other by changing the 0.5 threshold in `evaluate`.

### 6c. Look at individual predictions

Metrics summarise; it's still worth looking at examples. Below are the predictions the model was most confident about, right and wrong. Confident mistakes are worth a closer look: they can point to labelling errors in the data or to cases the model has misunderstood.

In [ ]:
results = pd.DataFrame({
    "sequence (start)": [s[:40] + "..." for s in TEST_SEQUENCES],
    "true": [LABELS[y] for y in TEST_LABELS],
    "predicted": [LABELS[y] for y in after_preds],
    "P(promoter)": after_probs.round(3),
})
results["confidence"] = np.abs(after_probs - 0.5)
correct = results["true"] == results["predicted"]

print("Most confident correct predictions:")
display(results[correct].nlargest(5, "confidence").drop(columns="confidence"))
print("Most confident mistakes:")
display(results[~correct].nlargest(5, "confidence").drop(columns="confidence"))

## 7. Save the LoRA adapter

As in Notebook 1, we save only what we trained: the adapters **and the classification head**. To use it later, load the base model and attach this adapter.

Colab deletes files when your session ends. Notebook 3 shows how to keep the adapter in Google Drive or on the Hugging Face Hub.

In [ ]:
model.save_pretrained("promoter_lora")
tokenizer.save_pretrained("promoter_lora")
!ls -lh promoter_lora

## Recap

- **The same LoRA method** worked on a completely different model and task. Only the loading, the data and the evaluation changed.
- **Data preparation:** check label balance, tokenise (DNA into 6-mers), and keep the test set untouched. Here it even comes from different chromosomes.
- **Baselines:** a random head scores like guessing. Training only the head is a cheap alternative worth trying before LoRA.
- **Evaluation:** with labelled data you can measure exactly. Use more than one metric (MCC is robust to unbalanced labels), and look at the mistakes, not just the scores.

**Next:** Notebook 3 shows how to keep your adapter and apply this to your own data.

> **DRAFT NOTE:** Optional extension for fast finishers: try the `promoter_tata` task (change `DATA_PATH`), change `r` or `N_TRAIN`, or add `"key"` to `target_modules` and compare MCC.